# Автоматическое горизонтальное выравнивание изображений

Ноутбук рекурсивно читает изображения из `data`, без использования разметки оценивает направление длинной оси центрального объекта и поворачивает его горизонтально.

Результаты сохраняются в `aligned_data` с той же структурой подпапок. Угол и диагностическая информация записываются в `aligned_data/alignment_report.csv`.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/font-cache")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
OUTPUT_DIR = Path("aligned_data")

# Уменьшенная копия используется только для поиска угла. Поворот делается на оригинале.
MAX_DETECTION_SIDE = 480
KMEANS_CLUSTERS = 4
USE_GRABCUT = False  # Можно включить для сложного фона, но обработка станет медленнее.
PREVIEW_COUNT = 10

# Если абсолютный угол меньше порога, изображение не поворачивается.
MIN_ROTATION_DEGREES = 0.35

# Сохранять диагностические маски рядом с результатами.
SAVE_DEBUG_MASKS = False

plt.rcParams["figure.dpi"] = 120
print(f"OpenCV: {cv2.__version__}")

## Поиск изображений и подготовка уменьшенной копии

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def collect_image_paths(data_dir: Path = DATA_DIR) -> list[Path]:
    return sorted(
        path for path in data_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )


def load_bgr(path: Path) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Не удалось прочитать изображение: {path}")
    return image


def resize_to_max_side(image: np.ndarray, max_side: int) -> tuple[np.ndarray, float]:
    height, width = image.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale == 1.0:
        return image.copy(), scale
    resized = cv2.resize(
        image,
        (int(round(width * scale)), int(round(height * scale))),
        interpolation=cv2.INTER_AREA,
    )
    return resized, scale


image_paths = collect_image_paths()
print(f"Найдено изображений: {len(image_paths)}")
image_paths[:5]

## Безразметочная оценка угла

Основной угол определяется по доминирующим длинным линиям в центральной части кадра. Если выраженные линии не найдены, алгоритм строит несколько независимых масок и вычисляет направление длинной оси наиболее подходящей области методом главных компонент (PCA).

In [ ]:
def _otsu_masks(image_bgr: np.ndarray) -> list[tuple[str, np.ndarray]]:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    _, bright = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return [("otsu_bright", bright), ("otsu_dark", cv2.bitwise_not(bright))]


def _kmeans_masks(image_bgr: np.ndarray) -> list[tuple[str, np.ndarray]]:
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    pixels = lab.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 25, 0.8)
    cv2.setRNGSeed(42)
    _, labels, centers = cv2.kmeans(
        pixels,
        KMEANS_CLUSTERS,
        None,
        criteria,
        1,
        cv2.KMEANS_PP_CENTERS,
    )
    labels = labels.reshape(image_bgr.shape[:2])
    result = []
    for cluster_id in range(KMEANS_CLUSTERS):
        result.append((f"kmeans_{cluster_id}", (labels == cluster_id).astype(np.uint8) * 255))

    # Корпус может состоять из двух близких оттенков, поэтому добавляем пары
    # соседних по яркости кластеров.
    order = np.argsort(centers[:, 0])
    for first, second in zip(order[:-1], order[1:]):
        mask = np.isin(labels, [first, second]).astype(np.uint8) * 255
        result.append((f"kmeans_pair_{first}_{second}", mask))
    return result


def _grabcut_mask(image_bgr: np.ndarray) -> tuple[str, np.ndarray]:
    height, width = image_bgr.shape[:2]
    margin_x = max(2, int(round(width * 0.025)))
    margin_y = max(2, int(round(height * 0.025)))
    rect = (margin_x, margin_y, width - 2 * margin_x, height - 2 * margin_y)
    mask = np.zeros((height, width), np.uint8)
    bg_model = np.zeros((1, 65), np.float64)
    fg_model = np.zeros((1, 65), np.float64)
    cv2.grabCut(image_bgr, mask, rect, bg_model, fg_model, 3, cv2.GC_INIT_WITH_RECT)
    foreground = np.where(
        (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0
    ).astype(np.uint8)
    return "grabcut", foreground


def candidate_masks(image_bgr: np.ndarray) -> list[tuple[str, np.ndarray]]:
    candidates = _otsu_masks(image_bgr)
    candidates.extend(_kmeans_masks(image_bgr))
    if USE_GRABCUT:
        try:
            candidates.append(_grabcut_mask(image_bgr))
        except cv2.error:
            pass
    return candidates

In [ ]:
def _clean_mask(mask: np.ndarray) -> np.ndarray:
    min_side = min(mask.shape)
    close_size = max(5, int(round(min_side * 0.025)))
    close_size += 1 - close_size % 2
    open_size = max(3, int(round(min_side * 0.008)))
    open_size += 1 - open_size % 2
    closed = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_size, close_size)),
    )
    return cv2.morphologyEx(
        closed,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_size, open_size)),
    )


def _component_metrics(component_mask: np.ndarray) -> dict | None:
    height, width = component_mask.shape
    image_area = height * width
    area = int(cv2.countNonZero(component_mask))
    area_ratio = area / image_area
    if not 0.012 <= area_ratio <= 0.88:
        return None

    contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    points = contour.reshape(-1, 2).astype(np.float32)
    if len(points) < 5:
        return None

    moments = cv2.moments(component_mask, binaryImage=True)
    center_x = moments["m10"] / max(moments["m00"], 1e-6)
    center_y = moments["m01"] / max(moments["m00"], 1e-6)
    center_distance = np.hypot(
        (center_x - width / 2) / (width / 2),
        (center_y - height / 2) / (height / 2),
    )

    rect = cv2.minAreaRect(contour)
    rect_area = max(rect[1][0] * rect[1][1], 1.0)
    hull_area = max(cv2.contourArea(cv2.convexHull(contour)), 1.0)
    rectangularity = min(1.0, area / rect_area)
    solidity = min(1.0, area / hull_area)

    # Для ориентации достаточно точек внешнего контура; это заметно быстрее,
    # чем собирать координаты каждого пикселя большой области.
    coords_yx = points[:, ::-1]
    covariance = np.cov(coords_yx, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    major_value = max(float(eigenvalues[order[0]]), 1e-6)
    minor_value = max(float(eigenvalues[order[1]]), 1e-6)
    elongation = float(np.sqrt(major_value / minor_value))

    # coords имеют порядок (y, x). Угол в координатах изображения нужен
    # непосредственно для cv2.getRotationMatrix2D.
    major_vector_yx = eigenvectors[:, order[0]]
    angle = float(np.degrees(np.arctan2(major_vector_yx[0], major_vector_yx[1])))
    angle = ((angle + 90.0) % 180.0) - 90.0

    x, y, box_width, box_height = cv2.boundingRect(contour)
    touched_sides = sum([
        x <= 1,
        y <= 1,
        x + box_width >= width - 1,
        y + box_height >= height - 1,
    ])

    center_score = max(0.0, 1.0 - center_distance / 1.1)
    area_score = min(1.0, area_ratio / 0.28)
    elongation_score = min(1.0, max(0.0, np.log2(elongation) / 1.8))
    score = (
        2.4 * center_score
        + 1.5 * area_score
        + 1.2 * rectangularity
        + 0.8 * solidity
        + 0.8 * elongation_score
        - 0.65 * touched_sides
    )
    return {
        "score": float(score),
        "angle": angle,
        "area_ratio": float(area_ratio),
        "center_distance": float(center_distance),
        "rectangularity": float(rectangularity),
        "solidity": float(solidity),
        "elongation": elongation,
        "mask": component_mask,
    }


def estimate_angle_from_lines(image_bgr: np.ndarray) -> dict | None:
    work_image, scale = resize_to_max_side(image_bgr, max(MAX_DETECTION_SIDE, 640))
    height, width = work_image.shape[:2]
    gray = cv2.cvtColor(work_image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(gray, 40, 120)

    # Края кадра часто дают сильные, но не относящиеся к объекту линии.
    margin = max(2, int(round(min(height, width) * 0.04)))
    edges[:margin] = 0
    edges[-margin:] = 0
    edges[:, :margin] = 0
    edges[:, -margin:] = 0

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=25,
        minLineLength=int(round(min(height, width) * 0.10)),
        maxLineGap=int(round(min(height, width) * 0.025)),
    )
    if lines is None or len(lines) < 2:
        return None

    weighted_angles = []
    for x1, y1, x2, y2 in lines[:, 0]:
        dx = float(x2 - x1)
        dy = float(y2 - y1)
        length = float(np.hypot(dx, dy))
        angle = ((float(np.degrees(np.arctan2(dy, dx))) + 90.0) % 180.0) - 90.0
        midpoint_x = (x1 + x2) / 2
        midpoint_y = (y1 + y2) / 2
        center_distance = np.hypot(
            (midpoint_x - width / 2) / (width / 2),
            (midpoint_y - height / 2) / (height / 2),
        )
        center_weight = max(0.10, 1.0 - center_distance / 1.4)
        weighted_angles.append((angle, length * length * center_weight))

    bins = np.arange(-90.0, 90.0, 5.0)
    support_by_bin = []
    for bin_angle in bins:
        support = sum(
            weight for angle, weight in weighted_angles
            if abs(((angle - bin_angle + 90.0) % 180.0) - 90.0) < 7.5
        )
        support_by_bin.append((support, bin_angle))
    best_support, best_bin = max(support_by_bin)
    nearby = [
        (angle, weight) for angle, weight in weighted_angles
        if abs(((angle - best_bin + 90.0) % 180.0) - 90.0) < 10.0
    ]
    vector = sum(weight * np.exp(2j * np.radians(angle)) for angle, weight in nearby)
    angle = float(np.degrees(np.angle(vector)) / 2.0)
    total_support = max(sum(weight for _, weight in weighted_angles), 1e-6)
    return {
        "angle": angle,
        "method": "hough_lines",
        "score": float(best_support / total_support),
        "line_count": int(len(lines)),
        "area_ratio": np.nan,
        "center_distance": np.nan,
        "rectangularity": np.nan,
        "solidity": np.nan,
        "elongation": np.nan,
        "mask": edges,
        "scale": scale,
    }


def estimate_horizontal_angle(image_bgr: np.ndarray) -> dict:
    line_result = estimate_angle_from_lines(image_bgr)
    if line_result is not None:
        return line_result

    work_image, scale = resize_to_max_side(image_bgr, MAX_DETECTION_SIDE)
    best = None

    for method, raw_mask in candidate_masks(work_image):
        clean_mask = _clean_mask(raw_mask)
        count, labels, stats, _ = cv2.connectedComponentsWithStats(clean_mask, connectivity=8)
        for component_id in range(1, count):
            area_ratio = stats[component_id, cv2.CC_STAT_AREA] / clean_mask.size
            if area_ratio < 0.012:
                continue
            component_mask = (labels == component_id).astype(np.uint8) * 255
            metrics = _component_metrics(component_mask)
            if metrics is None:
                continue
            metrics["method"] = method
            metrics["scale"] = scale
            if best is None or metrics["score"] > best["score"]:
                best = metrics

    if best is None:
        raise RuntimeError("Не удалось найти подходящую область для оценки угла")
    return best

## Поворот без обрезания и предпросмотр

In [ ]:
def estimate_border_color(image_bgr: np.ndarray) -> tuple[int, int, int]:
    height, width = image_bgr.shape[:2]
    strip = max(1, int(round(min(height, width) * 0.02)))
    border_pixels = np.concatenate([
        image_bgr[:strip].reshape(-1, 3),
        image_bgr[-strip:].reshape(-1, 3),
        image_bgr[:, :strip].reshape(-1, 3),
        image_bgr[:, -strip:].reshape(-1, 3),
    ])
    return tuple(int(value) for value in np.median(border_pixels, axis=0))


def rotate_bound(image_bgr: np.ndarray, angle: float) -> np.ndarray:
    if abs(angle) < MIN_ROTATION_DEGREES:
        return image_bgr.copy()

    height, width = image_bgr.shape[:2]
    center = (width / 2, height / 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cosine = abs(matrix[0, 0])
    sine = abs(matrix[0, 1])
    new_width = int(np.ceil(height * sine + width * cosine))
    new_height = int(np.ceil(height * cosine + width * sine))
    matrix[0, 2] += new_width / 2 - center[0]
    matrix[1, 2] += new_height / 2 - center[1]
    return cv2.warpAffine(
        image_bgr,
        matrix,
        (new_width, new_height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=estimate_border_color(image_bgr),
    )


def align_image(image_bgr: np.ndarray) -> tuple[np.ndarray, dict]:
    info = estimate_horizontal_angle(image_bgr)
    aligned = rotate_bound(image_bgr, info["angle"])
    return aligned, info


def show_preview(paths: list[Path], count: int = PREVIEW_COUNT) -> None:
    selected = paths[:count]
    if not selected:
        print("Нет изображений для предпросмотра")
        return

    fig, axes = plt.subplots(len(selected), 2, figsize=(11, 4 * len(selected)), squeeze=False)
    for row, path in enumerate(selected):
        original = load_bgr(path)
        aligned, info = align_image(original)
        axes[row, 0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
        axes[row, 0].set_title(f"До: {path.relative_to(DATA_DIR)}")
        axes[row, 1].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
        axes[row, 1].set_title(
            f"После: поворот {info['angle']:+.1f}°, метод {info['method']}"
        )
        for axis in axes[row]:
            axis.axis("off")
    plt.tight_layout()
    plt.show()


show_preview(image_paths)

## Обработка всей папки

Ячейка ниже создаёт `aligned_data`, повторяет структуру `data` и сохраняет все выровненные изображения.

In [ ]:
def save_image(path: Path, image_bgr: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()
    params = [cv2.IMWRITE_JPEG_QUALITY, 95] if suffix in {".jpg", ".jpeg"} else []
    if not cv2.imwrite(str(path), image_bgr, params):
        raise OSError(f"Не удалось сохранить изображение: {path}")


def process_all(paths: list[Path]) -> pd.DataFrame:
    records = []
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for index, input_path in enumerate(paths, start=1):
        relative_path = input_path.relative_to(DATA_DIR)
        output_path = OUTPUT_DIR / relative_path
        try:
            original = load_bgr(input_path)
            aligned, info = align_image(original)
            save_image(output_path, aligned)

            if SAVE_DEBUG_MASKS:
                debug_path = OUTPUT_DIR / "_debug_masks" / relative_path.with_suffix(".png")
                debug_path.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(debug_path), info["mask"])

            records.append({
                "input": str(input_path),
                "output": str(output_path),
                "angle_degrees": round(info["angle"], 4),
                "method": info["method"],
                "score": round(info["score"], 4),
                "area_ratio": round(info["area_ratio"], 4),
                "elongation": round(info["elongation"], 4),
                "line_count": info.get("line_count", 0),
                "status": "ok",
            })
        except Exception as error:
            records.append({
                "input": str(input_path),
                "output": str(output_path),
                "status": "error",
                "error": str(error),
            })
        if index % 10 == 0 or index == len(paths):
            print(f"Обработано: {index}/{len(paths)}")

    report = pd.DataFrame(records)
    report.to_csv(OUTPUT_DIR / "alignment_report.csv", index=False)
    return report


report = process_all(image_paths)
report

## Сводка результата

Эта ячейка показывает ошибки обработки и изображения с самыми большими поворотами. Проверка использует только отчёт самого алгоритма и не обращается к разметке.

In [ ]:
print(report["status"].value_counts(dropna=False))
display(report[report["status"] != "ok"])
display(
    report[report["status"] == "ok"]
    .assign(abs_angle=lambda frame: frame["angle_degrees"].abs())
    .sort_values("abs_angle", ascending=False)
    .head(20)
)